# MLP MNIST Classifier For Hand-drawn Digits

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data  import DataLoader
from torchvision import datasets, transforms

In [3]:
torch.manual_seed(42)

transformer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainData = datasets.MNIST(root='./data', train=True, download=True, transform=transformer)
testData = datasets.MNIST(root='./data', train=False, download=True, transform=transformer)

train_loader = DataLoader(trainData, batch_size=64, shuffle=True)
test_loader = DataLoader(testData, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.90MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 116kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.10MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.9MB/s]


In [4]:
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
        
    def forward(self, x):
        return self.layers(x)
        

In [15]:
model = MyNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
for epoch in range(6):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 Avg Loss: 0.2690
Epoch 2 Avg Loss: 0.1118
Epoch 3 Avg Loss: 0.0782
Epoch 4 Avg Loss: 0.0608
Epoch 5 Avg Loss: 0.0495
Epoch 6 Avg Loss: 0.0388


In [17]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        prediction = output.argmax(dim=1)
        correct += (prediction == y_batch).sum().item()
        total += y_batch.size(0)
        
print(f"\nTest accuracy: {100 * correct / total:.2f}%")
        


Test accuracy: 97.58%


In [22]:
PATH = "MNIST_model_weights.pth"
torch.save(model.state_dict(), PATH)
print(f"Model state_dict saved successfully to {PATH}")

Model state_dict saved successfully to MNIST_model_weights.pth
